# Notebook 5 — Fine-Tuning Pipeline

This is the most advanced part of the project. You will:

1. Collect historical PR review data from a GitHub repository
2. Understand the JSONL format OpenAI requires for fine-tuning
3. Validate the training data quality
4. Upload the data and start a fine-tuning job
5. Monitor training progress
6. Switch the reviewer over to your fine-tuned model automatically
7. Compare outputs: base model vs fine-tuned model

**Why fine-tune?** The base model gives general code review advice. A fine-tuned model learns the *specific patterns, terminology, and standards* of your team's codebase.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')

import json
import pathlib
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from app.data_collector import PRDataCollector, ReviewExample
from app.fine_tuner import FineTuner
from app.model_registry import get_active_model, set_active_model, clear_fine_tuned_model
from app.config import Config

print('Environment loaded.')

## Step 1 — Collect Historical PR Data

We fetch closed pull requests and extract the human reviewer's comments as training labels.

**Input:**  PR diff (what changed)
**Output:** Human review comment (what a skilled reviewer said about it)

We'll use the public `pallets/flask` repo for this demo. For your own projects, swap in your org/repo.

In [ ]:
collector = PRDataCollector()

# For a quick demo, collect 20 PRs. For production, aim for 50–200.
examples = collector.collect(
    owner='pallets',
    repo='flask',
    max_prs=20,
    rate_limit_sleep=0.5,
)

stats = PRDataCollector.summarise(examples)
print(f"Collected {stats['count']} usable examples")
print(f"Average diff length : {stats['avg_diff_chars']:,} characters")
print(f"Average review length: {stats['avg_review_chars']:,} characters")
print(f"By score hint       : {stats['by_score']}")

## Step 2 — Inspect the Data Quality

In [ ]:
# Show a few examples
for ex in examples[:3]:
    print('='*70)
    print(f"PR #{ex.pr_number}: {ex.pr_title}")
    print(f"Reviewer: {ex.reviewer}  |  Score hint: {ex.score}")
    print(f"\nDiff preview (first 300 chars):\n{ex.diff[:300]}")
    print(f"\nReview comment:\n{ex.review_body[:400]}")
    print()

In [ ]:
# Visualise review length distribution
df = pd.DataFrame({
    'review_len': [len(e.review_body) for e in examples],
    'diff_len':   [len(e.diff)        for e in examples],
    'reviewer':   [e.reviewer          for e in examples],
})

fig = px.scatter(
    df, x='diff_len', y='review_len',
    hover_data=['reviewer'],
    title='Diff Length vs Review Length',
    labels={'diff_len': 'Diff (chars)', 'review_len': 'Review (chars)'},
    template='plotly_dark',
)
fig.show()

## Step 3 — Understand the JSONL Format

OpenAI fine-tuning requires data in **JSONL** (one JSON object per line).

Each record is a complete chat conversation:
- **system** → The same system prompt your production bot uses
- **user** → The diff the model will see at inference time
- **assistant** → The ideal response we want the model to produce

In [ ]:
ft = FineTuner()

# Preview one formatted record
sample_record = FineTuner._to_chat_record(examples[0])
print('JSONL record structure:')
print(json.dumps({
    'messages': [
        {'role': m['role'], 'content': m['content'][:120] + '...'}
        for m in sample_record['messages']
    ]
}, indent=2))

In [ ]:
# Write the JSONL files
import pathlib
OUT_DIR = pathlib.Path('../finetune_data')
OUT_DIR.mkdir(exist_ok=True)

split = max(1, int(len(examples) * 0.15))
train_examples = examples[split:]
val_examples   = examples[:split]

train_file = OUT_DIR / 'training.jsonl'
val_file   = OUT_DIR / 'validation.jsonl'

ft.write_jsonl(train_examples, train_file)
ft.write_jsonl(val_examples,   val_file)

print(f'Training   : {len(train_examples)} examples → {train_file}')
print(f'Validation : {len(val_examples)} examples → {val_file}')

# Verify it's valid JSONL
lines = train_file.read_text().strip().split('\n')
for line in lines[:3]:
    obj = json.loads(line)
    assert len(obj['messages']) == 3
print('\n✅ JSONL format valid!')

## Step 4 — Estimate Cost

OpenAI charges per token for fine-tuning. Let's estimate before we spend money.

In [ ]:
# Rough token estimate (1 token ≈ 4 characters)
total_chars = sum(
    sum(len(m['content']) for m in json.loads(line)['messages'])
    for line in train_file.read_text().strip().split('\n')
)
approx_tokens = total_chars // 4

# gpt-4o-mini fine-tuning: $0.003 per 1K tokens (training), $0.012/1K (inference)
COST_PER_1K_TRAIN = 0.003
n_epochs = 3
estimated_cost = (approx_tokens / 1000) * COST_PER_1K_TRAIN * n_epochs

print(f'Approximate tokens   : {approx_tokens:,}')
print(f'Training epochs      : {n_epochs}')
print(f'Estimated cost       : ${estimated_cost:.4f}')
print(f'\n(Inference cost is $0.012 per 1K tokens — same billing as regular API calls)')

## Step 5 — Upload & Start Fine-Tuning

⚠️ **This cell will charge your OpenAI account.** Run it only when you're ready.

Uncomment the code to proceed.

In [ ]:
# ── UNCOMMENT TO RUN ──────────────────────────────────────────────
# result = ft.run_pipeline(examples, n_epochs=3)
# print('Job started!')
# print('Job ID            :', result['job_id'])
# print('Training file ID  :', result['train_file_id'])
# print('Fine-tuned model  :', result['fine_tuned_model'])
# ──────────────────────────────────────────────────────────────────

print('Cell ready — uncomment to start fine-tuning.')
print('Fine-tuning typically takes 15–60 minutes depending on dataset size.')

## Step 6 — Monitor Training Progress

In [ ]:
# List all your fine-tuning jobs
jobs = ft.list_jobs()

if not jobs:
    print('No fine-tuning jobs found yet.')
else:
    for job in jobs:
        status_emoji = {'succeeded':'✅','failed':'❌','running':'⏳','queued':'🕐'}.get(job['status'],'⚪')
        print(f"{status_emoji} {job['id']}")
        print(f"   Status : {job['status']}")
        print(f"   Model  : {job['model'] or 'in progress'}")
        print(f"   Tokens : {job['trained_tokens'] or 'N/A'}")
        print()

In [ ]:
# Show training log events for a specific job (replace with your job ID)
# JOB_ID = 'ftjob-xxxxxxxxxxxx'
# events = ft.get_job_events(JOB_ID)
# for e in events:
#     print(f"[{e['level']}] {e['message']}")

## Step 7 — Switch to Your Fine-Tuned Model

Once the job succeeds, activate the fine-tuned model with one line.

The `AIReviewer` automatically picks it up — no code changes needed.

In [ ]:
# Check what model is currently active
active = get_active_model(Config.AI_MODEL)
print(f'Currently active model: {active}')

# To activate a fine-tuned model:
# set_active_model('ft:gpt-4o-mini-2024-07-18:your-org:your-suffix:abc123')

# To revert to the base model:
# clear_fine_tuned_model()

## Step 8 — Compare: Base Model vs Fine-Tuned Model

Run the same diff through both models and compare the output style.

In [ ]:
import openai
from app.ai_reviewer import AIReviewer

TEST_DIFF = '''
### File: database.py (modified)
@@ -5,8 +5,14 @@
+def get_user(user_id):
+    conn = sqlite3.connect('app.db')
+    cursor = conn.cursor()
+    result = cursor.execute(f"SELECT * FROM users WHERE id={user_id}").fetchone()
+    return result
'''

# Base model review
base_reviewer = AIReviewer()
# Temporarily override to base model for comparison
original_model = get_active_model(Config.AI_MODEL)
clear_fine_tuned_model()

print('Reviewing with BASE model...')
base_review = base_reviewer.review_diff(TEST_DIFF, pr_title='Add get_user function')
print(f'Base model score  : {base_review.get("score")}/10')
print(f'Issues found      : {len(base_review.get("issues", []))}')
display(Markdown('**Base model summary:** ' + base_review.get('summary', '')))

In [ ]:
# Fine-tuned model review (only run if you have a fine-tuned model)
FINE_TUNED_MODEL_ID = ''  # paste your model ID here, e.g. 'ft:gpt-4o-mini:...'

if FINE_TUNED_MODEL_ID:
    set_active_model(FINE_TUNED_MODEL_ID)
    ft_reviewer = AIReviewer()
    print('Reviewing with FINE-TUNED model...')
    ft_review = ft_reviewer.review_diff(TEST_DIFF, pr_title='Add get_user function')
    print(f'Fine-tuned model score  : {ft_review.get("score")}/10')
    print(f'Issues found            : {len(ft_review.get("issues", []))}')
    display(Markdown('**Fine-tuned model summary:** ' + ft_review.get('summary', '')))
else:
    print('Set FINE_TUNED_MODEL_ID above to compare outputs.')

## Summary

You now have a complete fine-tuning pipeline:

| Step | What happens |
|---|---|
| Data collection | Fetch real PR review comments from GitHub |
| Quality filtering | Remove bots, generic comments, and trivially short examples |
| JSONL formatting | Convert to OpenAI's chat fine-tuning format |
| Cost estimation | Predict spend before committing |
| Training | Upload to OpenAI and start a supervised fine-tuning job |
| Activation | One-line model switch with automatic fallback |
| Comparison | Validate the fine-tuned model produces better domain-specific output |

**Key insight for your resume:** This demonstrates understanding of the *full ML lifecycle* — data collection, formatting, training, deployment, and evaluation — not just calling an API.